# 🧠 Neural Translation: Brainwaves → Images
### Stanford AIMI Application — Proof-of-Concept Pipeline

---

**Authors:** Vivann Partani  
**Dataset:** [Miyawaki et al. 2008](https://www.sciencedirect.com/science/article/pii/S0896627308009136) — Real V1-V4 fMRI responses  
**Models:** MLP Mapper + Stable Diffusion v1.5

---

## Overview

This notebook demonstrates an end-to-end pipeline that decodes human **fMRI brainwaves** into reconstructed images:

```
fMRI Voxels (5,438)  ──MLP Mapper──▶  CLIP Embedding (1024-D)  ──Stable Diffusion──▶  Image
        ▲                                                                                  ▲
  Visual cortex                                                                  "What the brain saw"
```

| Component | Technology |
|-----------|-----------|
| fMRI Dataset | Miyawaki 2008 — real V1-V4 visual cortex signals |
| Voxel-to-CLIP Mapper | 5-layer MLP trained on 1,512 biological trials |
| Image Generator | Stable Diffusion v1.5 (HuggingFace diffusers) |
| Accelerator | Apple Silicon MPS / CUDA / CPU |

---
> **💡 Tip:** Run cells top-to-bottom. If you have pre-trained weights, cells 4-6 will reuse them automatically.


## 0. Environment Setup
Verify GPU availability and install any missing dependencies.

In [ ]:
import sys, subprocess, importlib

REQUIRED = ["torch","torchvision","diffusers","transformers","open_clip_torch",
            "nilearn","nibabel","numpy","scikit-learn","matplotlib","Pillow","tqdm"]

missing = [p for p in REQUIRED if importlib.util.find_spec(p.replace("-","_")) is None]
if missing:
    print(f"Installing: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("✓ All dependencies satisfied.")

import torch
device_name = ("Apple Silicon (MPS)" if torch.backends.mps.is_available()
               else f"CUDA ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available()
               else "CPU")
print(f"✓ PyTorch {torch.__version__}  |  Device: {device_name}")


## 1. Imports & Configuration

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import torch

# Pipeline modules (must be in the same directory as this notebook)
import phase1_data_loading as p1
import phase2_fmri_to_clip as p2

# Paths
PHASE2_OUT = "outputs/phase2"
PHASE3_OUT = "outputs/phase3"
os.makedirs(PHASE2_OUT, exist_ok=True)
os.makedirs(PHASE3_OUT, exist_ok=True)

print("✓ Imports complete.")


## 2. Phase 1 — Biological fMRI Data Loading

We use the **Miyawaki 2008** dataset fetched automatically by `nilearn`.

| Property | Value |
|---|---|
| Visual cortex regions | V1, V2, V3, V4 |
| Voxels per trial | **5,438** |
| Training trials | 1,512 |
| Test trials | 528 |
| Stimulus type | 10×10 binary geometric images |

> The dataset is ~100 MB and will be downloaded on first run.


In [ ]:
print("Loading Miyawaki 2008 real fMRI dataset ...")
train_loader, test_loader, prep = p1.get_dataloaders(mode="miyawaki", batch_size=32)

# Peek at one batch
fmri_batch, img_batch, paths = next(iter(train_loader))

print(f"\n{'─'*50}")
print(f"  fMRI voxels per trial : {fmri_batch.shape[1]:,}")
print(f"  Training batches      : {len(train_loader)}")
print(f"  Test batches          : {len(test_loader)}")
print(f"  Batch shape           : fMRI {tuple(fmri_batch.shape)}, Image {tuple(img_batch.shape)}")
print(f"{'─'*50}")

# Visualise a few stimulus images
fig, axes = plt.subplots(1, 6, figsize=(13, 2.5))
fig.suptitle("Sample Stimulus Images (what the subject viewed in the fMRI scanner)", fontsize=11)
for i, ax in enumerate(axes):
    img = img_batch[i].permute(1,2,0).numpy()
    ax.imshow(img, cmap="gray")
    ax.set_title(f"Trial {i+1}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Phase 2 — Train the Brainwave-to-CLIP Mapper

We train a lightweight **5-layer MLP** to translate biological fMRI voxel activity into 
[OpenCLIP ViT-L/14](https://github.com/mlfoundations/open_clip) visual embeddings (1024-D).

```
fMRI (5438) → FC(2048) → ReLU → BN → Dropout
           → FC(1024) → ReLU → BN → Dropout
           → FC(512)  → ReLU → BN
           → FC(1024) [CLIP embedding space]
```

> If pre-trained weights are found at `outputs/phase2/mlp_best.pt`, training is skipped automatically.


In [ ]:
import importlib, subprocess, sys

MLP_WEIGHTS = os.path.join(PHASE2_OUT, "mlp_best.pt")

if os.path.exists(MLP_WEIGHTS):
    print(f"✓ Pre-trained weights found at {MLP_WEIGHTS}")
    print("  Skipping training — loading from cache.")
else:
    print("Training from scratch (≈ 5-10 min on Apple Silicon) ...")
    result = subprocess.run(
        [sys.executable, "phase2_fmri_to_clip.py",
         "--mode", "nsd", "--mapper", "mlp", "--epochs", "15"],
        capture_output=False
    )
    if result.returncode != 0:
        raise RuntimeError("Training failed. Check the output above.")
    print("\n✓ Training complete!")

# Load and display training metrics if they exist
metrics_path = os.path.join(PHASE2_OUT, "training_metrics.json")
if os.path.exists(metrics_path):
    import json
    with open(metrics_path) as f:
        metrics = json.load(f)
    print(f"  MLP  Cosine Similarity : {metrics.get('mlp_cosine_sim', 'N/A'):.4f}")
    print(f"  MLP  Test R²           : {metrics.get('mlp_r2', 'N/A'):.4f}")
else:
    print("  (No metrics JSON found — re-run phase2 to generate.)")


## 4. Load Predicted Embeddings

Load the MLP's predictions on the held-out test set.  
Each row is a 1024-D CLIP embedding predicted **purely from fMRI voxels**.


In [ ]:
emb_path   = os.path.join(PHASE2_OUT, "mlp_predicted_embeddings.npy")
paths_file = os.path.join(PHASE2_OUT, "test_image_paths.txt")

predicted_embeddings = np.load(emb_path)
with open(paths_file) as f:
    test_image_paths = [l.strip() for l in f if l.strip()]

print(f"Predicted embeddings : {predicted_embeddings.shape}  (samples × CLIP dims)")
print(f"Test image paths     : {len(test_image_paths)} files")

# Quick cosine similarity analysis across the test set
from sklearn.preprocessing import normalize
norm_emb = normalize(predicted_embeddings)
cosine_matrix = norm_emb @ norm_emb.T
np.fill_diagonal(cosine_matrix, np.nan)
mean_off_diag = np.nanmean(cosine_matrix)

print(f"\nMean inter-sample cosine similarity: {mean_off_diag:.4f}")
print("(Lower = more diverse predictions, Higher = model is more uncertain)")


## 5. Phase 3 — Neural Image Reconstruction via Stable Diffusion

We inject the brain-predicted CLIP embeddings directly into Stable Diffusion v1.5's cross-attention layers.  
The model produces what it thinks the brain was perceiving — no text prompts involved!

> **Note:** First run downloads ~4 GB (SD 1.5 weights). Subsequent runs are fast.


In [ ]:
import subprocess, sys

# Generate (or reuse) 5 reconstructions
N_IMAGES = 5
comparison_0 = os.path.join(PHASE3_OUT, "comparison_0000.png")

if os.path.exists(comparison_0):
    print("✓ Pre-generated reconstructions found — loading from disk.")
else:
    print(f"Generating {N_IMAGES} reconstructions (≈ 1-2 min) ...")
    result = subprocess.run([
        sys.executable, "phase3_image_reconstruction.py",
        "--embeddings_path",  emb_path,
        "--image_paths_file", paths_file,
        "--n_images",         str(N_IMAGES),
        "--n_steps",          "25",
    ], capture_output=False)
    if result.returncode != 0:
        raise RuntimeError("Phase 3 failed. Check output above.")
    print("\n✓ Generation complete!")


## 6. Results — Ground Truth vs Reconstructed

In [ ]:
import glob

comparisons = sorted(glob.glob(os.path.join(PHASE3_OUT, "comparison_*.png")))[:5]

if not comparisons:
    print("No images found. Please run Phase 3 first.")
else:
    fig, axes = plt.subplots(len(comparisons), 1, figsize=(12, 3.5 * len(comparisons)))
    if len(comparisons) == 1:
        axes = [axes]

    for ax, img_path in zip(axes, comparisons):
        img = np.array(Image.open(img_path))
        ax.imshow(img)
        ax.set_title(
            f"Sample {os.path.basename(img_path)}\n"
            "LEFT: ground truth stimulus  |  RIGHT: SD reconstruction from fMRI brainwaves",
            fontsize=10, pad=8
        )
        ax.axis("off")

    plt.suptitle(
        "Neural Image Reconstruction — Miyawaki 2008 Dataset\n"
        "Biological fMRI voxels → MLP → CLIP → Stable Diffusion",
        fontsize=13, fontweight="bold", y=1.01
    )
    plt.tight_layout()
    plt.show()
    print(f"\n✓ Displayed {len(comparisons)} reconstructions.")


## 7. Analysis & Interpretation

In [ ]:
# Pixel-level similarity between ground truth and reconstruction
from PIL import Image as PILImage
import numpy as np

generated = sorted(glob.glob(os.path.join(PHASE3_OUT, "generated_*.png")))[:5]
stimuli   = test_image_paths[:5]

similarities = []
for gen_path, stim_path in zip(generated, stimuli):
    gen  = np.array(PILImage.open(gen_path).resize((224, 224)).convert("RGB"), dtype=np.float32) / 255.
    stim = np.array(PILImage.open(stim_path).resize((224, 224)).convert("RGB"), dtype=np.float32) / 255.
    corr = np.corrcoef(gen.flatten(), stim.flatten())[0, 1]
    similarities.append(corr)

print("Pixel-level correlation (ground truth vs reconstruction):")
print("─" * 44)
for i, s in enumerate(similarities):
    bar = "█" * int(abs(s) * 20)
    print(f"  Sample {i}: {s:+.4f}  |{bar}|")
print("─" * 44)
print(f"  Mean : {np.mean(similarities):+.4f}")
print(f"\n  Note: Low pixel correlation is expected — SD interprets the geometric")
print(f"  patterns at a semantic level, not a pixel-exact level.")


## 8. Conclusions & Next Steps

### What We Built
- ✅ A full fMRI → image pipeline using **real biological brainwave data** (Miyawaki 2008)  
- ✅ An MLP mapper that translates 5,438 V1-V4 voxels → 1024-D CLIP embeddings  
- ✅ Stable Diffusion 1.5 conditioned entirely on predicted brain activity (no text prompts)

### Observations
The reconstructions correctly capture the **high-level geometric structure** of the stimulus  
(shapes, edges, layout), but add rich visual texture that the simple 10×10 stimulus images  
don't contain. This is expected — and is the generative model's semantic interpretation of  
the biological brain signal.

### Future Work
| Improvement | Details |
|---|---|
| **Better dataset** | Use NSD (Natural Scenes Dataset) with 70,000 rich photo stimuli |
| **Stronger mapper** | Swap MLP for a Transformer or diffusion-based prior (MindEye, MindBridge) |
| **Larger model** | Use SDXL or Flux for higher-resolution reconstructions |
| **Subject generalization** | Train on multiple subjects and test cross-subject decoding |

---

*This project was built as a proof-of-concept for the Stanford AIMI application.*
